In [68]:
import pandas as pd

df = pd.read_pickle('../../data-reeval-multi/resmat.pkl')

In [69]:
lsat = df.iloc[:, df.columns.get_level_values("scenario") == 'lsat_qa'].dropna(how='any', axis=0)

In [70]:
df_res = pd.read_pickle('./lsat_qa_result.pkl')

In [71]:
b1 = 'If no batch of cookies is made on Wednesday, then which one of the following must be true?'
b2 = "If Kammer's audition is immediately before Yoshida's, which one of the following could be true?"
b3 = "If the first batch of peanut butter cookies is made on Tuesday, then each of the following could be true"
b4 = "If Julio leads the Thursday afternoon session, then for how many of the other lab assistants can one determine which sessions they lead?"
b5 = "Which one of the following could be an accurate matching of the hangers to the fabrics of the dresses that hang on them?"
a1 = "If no batch of cookies is made on Wednesday, then which one of the following must be true?"
a2 = "Which one of the following CANNOT be the second audition?"
a3 = "If the polyester dress is on hanger 2, then which one of the following must be true?"
a4 = "If Julio leads the Thursday afternoon session, then for how many"
a5 = "Which one of the following could be the schedule of deliveries to the schools, from the first to the fourth?"
c1 = "Which one of the following could be a complete and accurate list of the days on which the batches of each kind of cookie are made?"
c2 = "If the limousine is not serviced on Saturday, then each of the following could be true EXCEPT:"
c3 = "If all the other initial conditions remain in effect, which one of the following must be false?"
c4 = "Which one of the following, if substituted for the constraint that if Jaramillo is assigned"
c5 = "Which one of the following is a possible selection of employees for the team?"

In [72]:
df_res.columns.names

FrozenList(['input.text', 'scenario', 'benchmark'])

In [73]:
input_texts = df_res.columns.get_level_values('input.text')
a_questions = [a1, a2, a3, a4, a5]
b_questions = [b1, b2, b3, b4, b5]
c_questions = [c1, c2, c3, c4, c5]
target_questions = [q.strip() for q in a_questions + b_questions + c_questions]
normalized_texts = [text.strip() for text in input_texts]
matched_questions = [text for text in normalized_texts if text in target_questions]
len(matched_questions), len(set(matched_questions))

(0, 0)

In [74]:
next(text for text in normalized_texts if 'If no batch of cookies is made on Wednesday' in text)

'A bakery makes exactly three kinds of cookie—oatmeal, peanut butter, and sugar. Exactly three batches of each kind of cookie are made each week (Monday through Friday) and each batch is made, from start to finish, on a single day. The following conditions apply: No two batches of the same kind of cookie are made on the same day. At least one batch of cookies is made on Monday. The second batch of oatmeal cookies is made on the same day as the first batch of peanut butter cookies. The second batch of sugar cookies is made on Thursday.\nQuestion: If no batch of cookies is made on Wednesday, then which one of the following must be true?'

In [75]:
def match_question(input_text: str, question: str) -> bool:
    cleaned = input_text.strip()
    question_clean = question.strip()
    return cleaned.endswith(question_clean) or f"Question: {question_clean}" in cleaned

question_groups = {
    'a': [a1, a2, a3, a4, a5],
    'b': [b1, b2, b3, b4, b5],
    'c': [c1, c2, c3, c4, c5],
}

grouped_columns = {}
for label, questions in question_groups.items():
    matched_cols = [
        col for col in df_res.columns
        if any(match_question(col[0], question) for question in questions)
    ]
    grouped_columns[label] = matched_cols

{label: len(cols) for label, cols in grouped_columns.items()}

{'a': 5, 'b': 5, 'c': 5}

In [76]:
grouped_res = pd.concat({label: df_res.loc[:, cols] for label, cols in grouped_columns.items()}, axis=1)
grouped_res.columns.names

FrozenList([None, 'input.text', 'scenario', 'benchmark'])

In [77]:
grouped_res.columns = grouped_res.columns.set_names(['group', 'input.text', 'scenario', 'benchmark'])
grouped_res.columns.names

FrozenList(['group', 'input.text', 'scenario', 'benchmark'])

In [78]:
validation_map = {}
for label, cols in grouped_columns.items():
    matched_questions = set()
    for col in cols:
        input_text = col[0]
        for question in question_groups[label]:
            if match_question(input_text, question):
                matched_questions.add(question)
                break
    validation_map[label] = sorted(matched_questions)
validation_map

{'a': ['If Julio leads the Thursday afternoon session, then for how many',
  'If no batch of cookies is made on Wednesday, then which one of the following must be true?',
  'If the polyester dress is on hanger 2, then which one of the following must be true?',
  'Which one of the following CANNOT be the second audition?',
  'Which one of the following could be the schedule of deliveries to the schools, from the first to the fourth?'],
 'b': ['If Julio leads the Thursday afternoon session, then for how many of the other lab assistants can one determine which sessions they lead?',
  "If Kammer's audition is immediately before Yoshida's, which one of the following could be true?",
  'If no batch of cookies is made on Wednesday, then which one of the following must be true?',
  'If the first batch of peanut butter cookies is made on Tuesday, then each of the following could be true',
  'Which one of the following could be an accurate matching of the hangers to the fabrics of the dresses th

In [79]:
original_columns = list(df_res.columns)
column_index_map = {col: idx for idx, col in enumerate(original_columns)}

group_column_indices = {
    label: [
        {
            'index': column_index_map[col],
            'input.text': col[0],
            'scenario': col[1],
            'benchmark': col[2],
        }
        for col in cols
    ]
    for label, cols in grouped_columns.items()
}
group_column_indices

{'a': [{'index': 1,
   'input.text': 'A bakery makes exactly three kinds of cookie—oatmeal, peanut butter, and sugar. Exactly three batches of each kind of cookie are made each week (Monday through Friday) and each batch is made, from start to finish, on a single day. The following conditions apply: No two batches of the same kind of cookie are made on the same day. At least one batch of cookies is made on Monday. The second batch of oatmeal cookies is made on the same day as the first batch of peanut butter cookies. The second batch of sugar cookies is made on Thursday.\nQuestion: If no batch of cookies is made on Wednesday, then which one of the following must be true?',
   'scenario': 'lsat_qa',
   'benchmark': 'classic'},
  {'index': 7,
   'input.text': "A chemistry class has six lab sessions scheduled over three days—Wednesday, Thursday, and Friday—one session heing held each morning and one each afternoon. Each session will be led by a different lab assistant—Julio, Kevin, Lan, N

In [80]:
grouped_index_df = (
    pd.DataFrame(
        [
            {
                'group': label,
                'index': entry['index'],
                'input.text': entry['input.text'],
                'scenario': entry['scenario'],
                'benchmark': entry['benchmark'],
            }
            for label, entries in group_column_indices.items()
            for entry in entries
        ]
    )
    .sort_values('index')
    .reset_index(drop=True)
 )
grouped_index_df

,group,index,input.text,scenario,benchmark
0,a,1,A bakery makes exactly three kinds of cookie—o...,lsat_qa,classic
1,b,1,A bakery makes exactly three kinds of cookie—o...,lsat_qa,classic
2,b,3,A bakery makes exactly three kinds of cookie—o...,lsat_qa,classic
3,c,5,A bakery makes exactly three kinds of cookie—o...,lsat_qa,classic
4,a,7,A chemistry class has six lab sessions schedul...,lsat_qa,classic
5,b,7,A chemistry class has six lab sessions schedul...,lsat_qa,classic
6,b,11,A chorus director is planning to audition exac...,lsat_qa,classic
7,a,12,A chorus director is planning to audition exac...,lsat_qa,classic
8,c,17,"A closet contains exactly six hangers—1, 2, 3,...",lsat_qa,classic
9,a,19,"A closet contains exactly six hangers—1, 2, 3,...",lsat_qa,classic


In [108]:
print(grouped_index_df.groupby('group')['index'].apply(list))

group
a     [1, 7, 12, 19, 27]
b      [1, 3, 7, 11, 23]
c    [5, 17, 31, 53, 81]
Name: index, dtype: object


In [107]:
import numpy as np
int_array = lsat.values.astype(int)

# create a new array with the index as the leftmost column and the dataframe values to the right
index_col = lsat.index.to_numpy().astype(str)
combined = np.column_stack([index_col, int_array])

output_path = f'../../data-reeval-multi/lsat_qa'

np.savetxt(f"{output_path}/lsat_resmat.csv", combined, delimiter=',', fmt='%s')

# np.savetxt(output_path + '/lsat_resmat.csv', lsat.values, delimiter=',', fmt='%d')
# np.savetxt(f"{output_path}/lsat_column.csv", lsat.index.to_numpy(), delimiter=',', fmt='%s')